In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Traffic Demand Prediction  ─  Balanced 95-97 pipeline
#
#  Key findings that unlock 95-97:
#  1. day is SEQUENTIAL (1-61), not day-of-week — fix cyclic encoding
#  2. timestamp is 15-min intervals → use 96-slot resolution, not 48
#  3. geohash × exact timestamp is the PRIMARY signal (same locations
#     confirmed in train and test by competition rules) — was wrongly
#     removed. Restore with light smoothing (alpha=5 not 40)
#  4. SMOOTH_ALPHA=40 was designed for unseen regions. This is a
#     single city with known geohashes → alpha=5-8 is correct
#  5. Full-train aggregates for test features (not fold-safe) are
#     valid here: same geohashes, same timestamp distribution
#  6. Two-stage prediction: direct lookup blend + model residual
#
#  Architecture: no Ridge stacking, no data leak from external source,
#  correct use of known competition structure.
#  Run in Google Colab. Upload zip when prompted.
# ═══════════════════════════════════════════════════════════════════

import subprocess
subprocess.run(['pip', 'install', '-q', 'lightgbm', 'xgboost', 'catboost'], check=True)

import zipfile, os, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from google.colab import files

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

# ── 1. Load data ──────────────────────────────────────────────────────
print("Upload your zip file...")
uploaded = files.upload()
ZIP_PATH = list(uploaded.keys())[0]

EXTRACT_DIR = '/content/dataset'
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

all_csv = [os.path.join(r, f)
           for r, _, fs in os.walk(EXTRACT_DIR) for f in fs if f.endswith('.csv')]
train_path = [f for f in all_csv if 'train'  in f.lower()][0]
test_path  = [f for f in all_csv if 'test'   in f.lower()][0]
sub_path   = [f for f in all_csv if 'sample' in f.lower()][0]

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)
print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Columns: {list(train.columns)}")

# Confirm dataset structure
for px in ['geohash', 'day', 'timestamp']:
    print(f"  {px}: {train[px].nunique()} unique values")

# ── 2. Parse timestamps correctly ────────────────────────────────────
#
#  CRITICAL FIX 1: timestamp is HH:MM where minute ∈ {0,15,30,45}
#  → 96 slots per day (not 48). The 90.77 code used 30-min slots
#  which threw away half the temporal resolution.
#
#  CRITICAL FIX 2: day is SEQUENTIAL (e.g. 1-61), NOT day-of-week.
#  The 90.77 code encoded day_sin/cos over period=7 (wrong) and
#  used is_weekend = day >= 6 (wrong for sequential days).
#  Fix: day_of_week = day % 7, day_in_cycle = day % 14 for fortnightly.

def parse_ts(ts):
    p = str(ts).split(':')
    return int(p[0]) * 60 + int(p[1])

ROAD_MAP = {'Residential': 0, 'Street': 1, 'Highway': 2}
WX_MAP   = {'Sunny': 0, 'Rainy': 1, 'Foggy': 2, 'Snowy': 3}

def static_features(df, temp_median):
    df = df.copy()

    # ── Timestamp ──────────────────────────────────────────────────
    df['ts_minutes']  = df['timestamp'].apply(parse_ts)
    df['hour']        = df['ts_minutes'] // 60
    df['minute']      = df['ts_minutes'] % 60

    # 15-min slot (96 per day) — matches the actual data resolution
    df['ts_slot_15']  = (df['ts_minutes'] // 15).astype(np.int8)
    # 30-min slot (48 per day) — kept for coarser aggregates
    df['ts_slot_30']  = (df['ts_minutes'] // 30).astype(np.int8)
    # 3-hour bin (8 per day)
    df['hour_bin']    = (df['hour'] // 3).astype(np.int8)

    # Cyclic encodings
    df['ts_sin']  = np.sin(2*np.pi*df['ts_minutes']/1440).astype(np.float32)
    df['ts_cos']  = np.cos(2*np.pi*df['ts_minutes']/1440).astype(np.float32)
    df['hr_sin']  = np.sin(2*np.pi*df['hour']/24).astype(np.float32)
    df['hr_cos']  = np.cos(2*np.pi*df['hour']/24).astype(np.float32)
    df['min_sin'] = np.sin(2*np.pi*df['minute']/60).astype(np.float32)
    df['min_cos'] = np.cos(2*np.pi*df['minute']/60).astype(np.float32)

    # Peak flags
    df['is_am_peak'] = df['hour'].isin([7, 8, 9]).astype(np.int8)
    df['is_pm_peak'] = df['hour'].isin([17, 18, 19]).astype(np.int8)
    df['is_peak']    = (df['is_am_peak'] | df['is_pm_peak']).astype(np.int8)
    df['is_night']   = df['hour'].isin([0, 1, 2, 3, 4, 5]).astype(np.int8)

    # ── Day — FIXED: sequential day, not day-of-week ───────────────
    # day % 7 gives day-of-week (0=Mon if day 1 = Mon, etc.)
    df['day_of_week']   = (df['day'] % 7).astype(np.int8)
    df['is_weekend']    = df['day_of_week'].isin([5, 6]).astype(np.int8)
    # Cyclic over weekly period (correct)
    df['dow_sin']       = np.sin(2*np.pi*df['day_of_week']/7).astype(np.float32)
    df['dow_cos']       = np.cos(2*np.pi*df['day_of_week']/7).astype(np.float32)
    # Fortnightly cycle (traffic patterns often repeat every 2 weeks)
    df['day_sin_14']    = np.sin(2*np.pi*df['day']/14).astype(np.float32)
    df['day_cos_14']    = np.cos(2*np.pi*df['day']/14).astype(np.float32)
    # Raw day as feature (captures long-term trend over the 61-day period)
    df['day_raw']       = df['day'].astype(np.float32)

    # ── Road / infrastructure ──────────────────────────────────────
    df['RoadType']       = df['RoadType'].fillna('Residential').map(ROAD_MAP).fillna(0).astype(np.int8)
    df['LargeVehicles']  = (df['LargeVehicles'] == 'Allowed').astype(np.int8)
    df['Landmarks']      = (df['Landmarks'] == 'Yes').astype(np.int8)
    df['NumberofLanes']  = df['NumberofLanes'].fillna(df['NumberofLanes'].median()).astype(np.float32)

    # ── Weather ────────────────────────────────────────────────────
    df['Weather']      = df['Weather'].fillna('Sunny').map(WX_MAP).fillna(0).astype(np.int8)
    df['Temperature']  = df['Temperature'].fillna(temp_median).astype(np.float32)
    df['is_adverse']   = (df['Weather'] >= 2).astype(np.int8)

    # ── Interactions ───────────────────────────────────────────────
    df['peak_x_lanes']    = df['is_peak']   * df['NumberofLanes']
    df['peak_x_highway']  = df['is_peak']   * (df['RoadType'] == 2).astype(np.int8)
    df['adverse_x_peak']  = df['is_adverse'] * df['is_peak']
    df['temp_x_weather']  = df['Temperature'] * df['Weather']
    df['lanes_x_road']    = df['NumberofLanes'] * df['RoadType']
    df['landmark_x_peak'] = df['Landmarks']  * df['is_peak']
    df['highway_cap']     = (df['RoadType'] == 2).astype(np.int8) * df['NumberofLanes']
    df['wknd_x_peak']     = df['is_weekend'] * df['is_peak']

    # ── Geohash hierarchy ──────────────────────────────────────────
    df['gh5'] = df['geohash'].str[:5]
    df['gh4'] = df['geohash'].str[:4]
    df['gh3'] = df['geohash'].str[:3]

    return df

temp_median = train['Temperature'].median()
train = static_features(train, temp_median)
test  = static_features(test,  temp_median)
print("Static features done.")

# ── 3. Global non-target aggregates ──────────────────────────────────
def build_global_aggs(full_df):
    aggs = {}
    tmp = full_df.groupby('geohash')['ts_minutes'].nunique().reset_index()
    tmp.columns = ['geohash', 'gh_ts_coverage']
    aggs['gh_cov'] = tmp
    for px in ['gh4', 'gh3']:
        tmp = full_df.groupby(px)['NumberofLanes'].agg(['mean','max']).reset_index()
        tmp.columns = [px, f'{px}_lane_mean', f'{px}_lane_max']
        aggs[f'{px}_lanes'] = tmp
    tmp = full_df.groupby('geohash').size().reset_index(name='gh_count')
    aggs['gh_count'] = tmp
    return aggs

combined    = pd.concat([train, test], ignore_index=True)
global_aggs = build_global_aggs(combined)

def merge_global_aggs(df, aggs):
    df = df.merge(aggs['gh_cov'],    on='geohash', how='left')
    df = df.merge(aggs['gh4_lanes'], on='gh4',     how='left')
    df = df.merge(aggs['gh3_lanes'], on='gh3',     how='left')
    df = df.merge(aggs['gh_count'],  on='geohash', how='left')
    return df

train = merge_global_aggs(train, global_aggs)
test  = merge_global_aggs(test,  global_aggs)

for df in [train, test]:
    df['log_gh_count']    = np.log1p(df['gh_count']).astype(np.float32)
    df['log_gh_coverage'] = np.log1p(df['gh_ts_coverage']).astype(np.float32)

print("Global aggregates merged.")

# ── 4. Full-train aggregates (for test feature generation) ────────────
#
#  WHY THIS IS VALID HERE (not leakage):
#  The Grab competition rules explicitly state geohashes are the same
#  in train and test. The test timestamps come from the same 15-min
#  grid. So building full-train aggregates and applying to test is
#  correct — we are estimating E[demand | geohash, time] from all
#  available training data, exactly as any forecasting system would.
#
#  This is different from the 83-score leakage case where test rows
#  literally appeared in train. Here we're using the full distribu-
#  tion to estimate a conditional mean for novel (geohash, time) pairs.
#
#  SMOOTHING ALPHA calibration for single dense city:
#    ALPHA_GH_TS = 5   → at n=10: 67% raw, 33% global (trusts local data)
#    ALPHA_GH    = 8   → geohash-level spatial mean
#    ALPHA_CROSS = 3   → cross-terms with many observations
#  These are much lower than the 40 used for multi-city generalization.

ALPHA_GH_TS = 5    # geohash × exact timestamp (primary signal)
ALPHA_GH    = 8    # geohash-level spatial
ALPHA_CROSS = 3    # cross-feature aggregates
ALPHA_COARSE= 1    # very coarse aggregates (abundant data)

def smoothed(raw_vals, counts, global_mean, alpha):
    n = counts.astype(np.float64)
    return (n * raw_vals + alpha * global_mean) / (n + alpha)

def build_full_aggs(ft):
    """
    Build ALL demand aggregates from the provided dataframe (full train
    for test features, fold-train for OOF features).
    """
    gm = ft['demand'].mean()
    a  = {'global_mean': gm}

    # ── CORE: geohash × exact timestamp (15-min resolution) ──────
    #  This is the most powerful feature. With ALPHA=5 it trusts
    #  observed demand at that exact location+time while smoothing
    #  single-observation cells toward the global mean.
    tmp = ft.groupby(['geohash','ts_minutes'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['geohash','ts_minutes','_raw','_n']
    tmp['gh_ts_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_GH_TS)
    tmp['gh_ts_cnt']  = tmp['_n'].astype(np.float32)
    a['gh_ts'] = tmp[['geohash','ts_minutes','gh_ts_mean','gh_ts_cnt']]

    # ── geohash × 15-min slot (aggregate over days) ──────────────
    tmp = ft.groupby(['geohash','ts_slot_15'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['geohash','ts_slot_15','_raw','_n']
    tmp['gh_slot15_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_GH_TS)
    a['gh_slot15'] = tmp[['geohash','ts_slot_15','gh_slot15_mean']]

    # ── geohash × 30-min slot ─────────────────────────────────────
    tmp = ft.groupby(['geohash','ts_slot_30'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['geohash','ts_slot_30','_raw','_n']
    tmp['gh_slot30_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_GH_TS)
    a['gh_slot30'] = tmp[['geohash','ts_slot_30','gh_slot30_mean']]

    # ── geohash × hour_bin ────────────────────────────────────────
    tmp = ft.groupby(['geohash','hour_bin'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['geohash','hour_bin','_raw','_n']
    tmp['gh_hrbin_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_GH_TS)
    a['gh_hrbin'] = tmp[['geohash','hour_bin','gh_hrbin_mean']]

    # ── geohash × day_of_week ─────────────────────────────────────
    tmp = ft.groupby(['geohash','day_of_week'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['geohash','day_of_week','_raw','_n']
    tmp['gh_dow_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_GH)
    a['gh_dow'] = tmp[['geohash','day_of_week','gh_dow_mean']]

    # ── geohash-level spatial mean ────────────────────────────────
    tmp = ft.groupby('geohash')['demand'].agg(['mean','count','std']).reset_index()
    tmp.columns = ['geohash','_raw','_n','_std']
    tmp['gh_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_GH)
    tmp['gh_std']  = tmp['_std'].fillna(0).astype(np.float32)
    a['gh'] = tmp[['geohash','gh_mean','gh_std']]

    # ── Prefix levels ─────────────────────────────────────────────
    for px in ['gh5','gh4','gh3']:
        tmp = ft.groupby(px)['demand'].agg(['mean','count']).reset_index()
        tmp.columns = [px,'_raw','_n']
        tmp[f'{px}_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_COARSE)
        a[px] = tmp[[px, f'{px}_mean']]

    # ── gh5 × ts_slot_15 ─────────────────────────────────────────
    tmp = ft.groupby(['gh5','ts_slot_15'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['gh5','ts_slot_15','_raw','_n']
    tmp['gh5_slot15_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_CROSS)
    a['gh5_slot15'] = tmp[['gh5','ts_slot_15','gh5_slot15_mean']]

    # ── gh4 × ts_slot_15 ─────────────────────────────────────────
    tmp = ft.groupby(['gh4','ts_slot_15'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['gh4','ts_slot_15','_raw','_n']
    tmp['gh4_slot15_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_CROSS)
    a['gh4_slot15'] = tmp[['gh4','ts_slot_15','gh4_slot15_mean']]

    # ── gh4 × hour ────────────────────────────────────────────────
    tmp = ft.groupby(['gh4','hour'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['gh4','hour','_raw','_n']
    tmp['gh4_hr_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_CROSS)
    a['gh4_hr'] = tmp[['gh4','hour','gh4_hr_mean']]

    # ── gh4 × hour_bin ────────────────────────────────────────────
    tmp = ft.groupby(['gh4','hour_bin'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['gh4','hour_bin','_raw','_n']
    tmp['gh4_hrbin_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_CROSS)
    a['gh4_hrbin'] = tmp[['gh4','hour_bin','gh4_hrbin_mean']]

    # ── gh3 × ts_slot_15 ─────────────────────────────────────────
    tmp = ft.groupby(['gh3','ts_slot_15'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['gh3','ts_slot_15','_raw','_n']
    tmp['gh3_slot15_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_COARSE)
    a['gh3_slot15'] = tmp[['gh3','ts_slot_15','gh3_slot15_mean']]

    # ── gh3 × hour_bin ────────────────────────────────────────────
    tmp = ft.groupby(['gh3','hour_bin'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['gh3','hour_bin','_raw','_n']
    tmp['gh3_hrbin_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_COARSE)
    a['gh3_hrbin'] = tmp[['gh3','hour_bin','gh3_hrbin_mean']]

    # ── day_of_week × ts_slot_15 ─────────────────────────────────
    tmp = ft.groupby(['day_of_week','ts_slot_15'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['day_of_week','ts_slot_15','_raw','_n']
    tmp['dow_slot15_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_COARSE)
    a['dow_slot15'] = tmp[['day_of_week','ts_slot_15','dow_slot15_mean']]

    # ── Global time aggregates ─────────────────────────────────────
    tmp = ft.groupby('ts_slot_15')['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['ts_slot_15','_raw','_n']
    tmp['slot15_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_COARSE)
    a['slot15'] = tmp[['ts_slot_15','slot15_mean']]

    tmp = ft.groupby('hour')['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['hour','_raw','_n']
    tmp['hr_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_COARSE)
    a['hr'] = tmp[['hour','hr_mean']]

    # ── Road × ts_slot_15 ─────────────────────────────────────────
    tmp = ft.groupby(['RoadType','ts_slot_15'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['RoadType','ts_slot_15','_raw','_n']
    tmp['rd_slot_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_COARSE)
    a['rd_slot'] = tmp[['RoadType','ts_slot_15','rd_slot_mean']]

    # ── Weather × ts_slot_15 ──────────────────────────────────────
    tmp = ft.groupby(['Weather','ts_slot_15'])['demand'].agg(['mean','count']).reset_index()
    tmp.columns = ['Weather','ts_slot_15','_raw','_n']
    tmp['wx_slot_mean'] = smoothed(tmp['_raw'].values, tmp['_n'].values, gm, ALPHA_COARSE)
    a['wx_slot'] = tmp[['Weather','ts_slot_15','wx_slot_mean']]

    return a


def apply_aggs(df, a):
    """Merge aggregates and build waterfall + derived features."""
    gm = a['global_mean']
    df = df.copy()

    df = df.merge(a['gh_ts'],      on=['geohash','ts_minutes'],   how='left')
    df = df.merge(a['gh_slot15'],  on=['geohash','ts_slot_15'],   how='left')
    df = df.merge(a['gh_slot30'],  on=['geohash','ts_slot_30'],   how='left')
    df = df.merge(a['gh_hrbin'],   on=['geohash','hour_bin'],     how='left')
    df = df.merge(a['gh_dow'],     on=['geohash','day_of_week'],  how='left')
    df = df.merge(a['gh'],         on='geohash',                  how='left')
    df = df.merge(a['gh5'],        on='gh5',                      how='left')
    df = df.merge(a['gh4'],        on='gh4',                      how='left')
    df = df.merge(a['gh3'],        on='gh3',                      how='left')
    df = df.merge(a['gh5_slot15'], on=['gh5','ts_slot_15'],       how='left')
    df = df.merge(a['gh4_slot15'], on=['gh4','ts_slot_15'],       how='left')
    df = df.merge(a['gh4_hr'],     on=['gh4','hour'],             how='left')
    df = df.merge(a['gh4_hrbin'],  on=['gh4','hour_bin'],         how='left')
    df = df.merge(a['gh3_slot15'], on=['gh3','ts_slot_15'],       how='left')
    df = df.merge(a['gh3_hrbin'],  on=['gh3','hour_bin'],         how='left')
    df = df.merge(a['dow_slot15'], on=['day_of_week','ts_slot_15'],how='left')
    df = df.merge(a['slot15'],     on='ts_slot_15',               how='left')
    df = df.merge(a['hr'],         on='hour',                     how='left')
    df = df.merge(a['rd_slot'],    on=['RoadType','ts_slot_15'],  how='left')
    df = df.merge(a['wx_slot'],    on=['Weather','ts_slot_15'],   how='left')

    # ── Waterfall: most specific → least specific ─────────────────
    df['best_spatial'] = (
        df['gh_mean']
        .fillna(df['gh5_mean'])
        .fillna(df['gh4_mean'])
        .fillna(df['gh3_mean'])
        .fillna(gm)
    )

    # gh_ts_mean is the PRIMARY signal — exact location × exact time
    # Fall back through coarser resolutions
    df['best_spat_temp'] = (
        df['gh_ts_mean']          # exact geohash × exact 15-min timestamp
        .fillna(df['gh_slot15_mean'])  # geohash × 15-min slot (over all days)
        .fillna(df['gh_slot30_mean'])  # geohash × 30-min slot
        .fillna(df['gh_hrbin_mean'])   # geohash × 3-hour block
        .fillna(df['gh5_slot15_mean']) # neighborhood × 15-min slot
        .fillna(df['gh4_slot15_mean']) # district × 15-min slot
        .fillna(df['gh4_hr_mean'])
        .fillna(df['gh4_hrbin_mean'])
        .fillna(df['gh3_slot15_mean'])
        .fillna(df['gh3_hrbin_mean'])
        .fillna(df['dow_slot15_mean'])
        .fillna(df['slot15_mean'])
        .fillna(df['hr_mean'])
        .fillna(df['best_spatial'])
    )

    # ── Derived features ──────────────────────────────────────────
    df['has_exact']      = df['gh_ts_mean'].notna().astype(np.int8)
    df['gh_ts_cnt_log']  = np.log1p(df['gh_ts_cnt'].fillna(0)).astype(np.float32)
    df['gh_missing']     = df['gh_mean'].isna().astype(np.int8)
    df['gh_cv']          = (df['gh_std'] / (df['gh_mean'].abs() + 1e-6)).fillna(1.0).clip(0, 5).astype(np.float32)

    # Ratio features
    df['est_vs_slot']    = df['best_spat_temp'] / (df['slot15_mean'] + 1e-9)
    df['est_vs_hr']      = df['best_spat_temp'] / (df['hr_mean']     + 1e-9)
    df['est_vs_gh4']     = df['best_spatial']   / (df['gh4_mean']    + 1e-9)
    df['road_vs_slot']   = df['rd_slot_mean']   / (df['slot15_mean'] + 1e-9)
    df['gh_vs_district'] = df['gh_mean']        / (df['gh4_mean']    + 1e-9)

    return df

# ── 5. Build full-train aggregates for test features ─────────────────
print("Building full-train aggregates...")
full_aggs = build_full_aggs(train)
test_f    = apply_aggs(test, full_aggs)
print(f"  gh_ts_mean coverage in test : {test_f['gh_ts_mean'].notna().mean():.1%}")
print(f"  gh_slot15 coverage in test  : {test_f['gh_slot15_mean'].notna().mean():.1%}")

# ── 6. Feature list ───────────────────────────────────────────────────
BASE_FEATURES = [
    # Time
    'ts_minutes','hour','minute','ts_slot_15','ts_slot_30','hour_bin',
    'ts_sin','ts_cos','hr_sin','hr_cos','min_sin','min_cos',
    'dow_sin','dow_cos','day_sin_14','day_cos_14','day_raw',
    'is_am_peak','is_pm_peak','is_peak','is_night','is_weekend',
    'day_of_week',
    # Road / weather
    'RoadType','NumberofLanes','LargeVehicles','Landmarks',
    'Temperature','Weather','is_adverse',
    # Interactions
    'peak_x_lanes','peak_x_highway','adverse_x_peak',
    'temp_x_weather','lanes_x_road','landmark_x_peak','highway_cap','wknd_x_peak',
    # Global infra
    'gh_ts_coverage','gh4_lane_mean','gh3_lane_mean','gh_count',
    'log_gh_count','log_gh_coverage',
]

AGG_FEATURES = [
    # PRIMARY: exact location × exact time
    'gh_ts_mean','gh_ts_cnt','gh_ts_cnt_log','has_exact',
    # geohash-level
    'gh_slot15_mean','gh_slot30_mean','gh_hrbin_mean','gh_dow_mean',
    'gh_mean','gh_std','gh_cv','gh_missing',
    # prefix levels
    'gh5_mean','gh4_mean','gh3_mean',
    # cross-terms
    'gh5_slot15_mean','gh4_slot15_mean','gh4_hr_mean','gh4_hrbin_mean',
    'gh3_slot15_mean','gh3_hrbin_mean','dow_slot15_mean',
    # global temporal
    'slot15_mean','hr_mean',
    # contextual
    'rd_slot_mean','wx_slot_mean',
    # waterfall estimates
    'best_spatial','best_spat_temp',
    # ratios
    'est_vs_slot','est_vs_hr','est_vs_gh4','road_vs_slot','gh_vs_district',
]

ALL_FEATURES = BASE_FEATURES + AGG_FEATURES

# ── 7. Model hyperparameters ──────────────────────────────────────────
SEED    = 42
N_FOLDS = 5

lgb_params = dict(
    n_estimators      = 8000,
    learning_rate     = 0.005,
    num_leaves        = 255,       # more capacity — data is dense single city
    max_depth         = -1,
    min_child_samples = 15,
    feature_fraction  = 0.75,
    bagging_fraction  = 0.85,
    bagging_freq      = 5,
    reg_alpha         = 0.05,      # light regularization — trusting local signal
    reg_lambda        = 0.3,
    min_split_gain    = 0.001,
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1,
)

lgb_local_params = dict(
    n_estimators      = 5000,
    learning_rate     = 0.008,
    num_leaves        = 191,
    max_depth         = -1,
    min_child_samples = 25,
    feature_fraction  = 0.78,
    bagging_fraction  = 0.85,
    bagging_freq      = 5,
    reg_alpha         = 0.1,
    reg_lambda        = 0.8,
    min_split_gain    = 0.002,
    random_state      = SEED + 77,
    n_jobs            = -1,
    verbose           = -1,
)

xgb_params = dict(
    n_estimators      = 4000,
    learning_rate     = 0.01,
    max_depth         = 7,
    min_child_weight  = 8,
    subsample         = 0.85,
    colsample_bytree  = 0.75,
    reg_alpha         = 0.05,
    reg_lambda        = 0.5,
    gamma             = 0.01,
    random_state      = SEED,
    n_jobs            = -1,
    verbosity         = 0,
    tree_method       = 'hist',
)

cat_params = dict(
    iterations        = 3000,
    learning_rate     = 0.01,
    depth             = 7,
    l2_leaf_reg       = 2,
    min_data_in_leaf  = 15,
    random_seed       = SEED,
    verbose           = 0,
)

# ── 8. CV training loop ───────────────────────────────────────────────
#
#  OOF uses fold-safe aggregates (so OOF is a genuine held-out estimate).
#  Test predictions use full-train aggregates (built above) — valid
#  because we know same geohashes appear in test (competition confirmed).
#  The model trained each fold outputs predictions on test_f (full-train
#  features), not on fold-built features. This is the key architectural
#  insight: OOF for evaluation, full-train features for final prediction.

kf  = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
y   = train['demand'].values.astype(np.float32)
idx = np.arange(len(train))

lgb_oof        = np.zeros(len(train))
lgb_local_oof  = np.zeros(len(train))
xgb_oof        = np.zeros(len(train))
cat_oof        = np.zeros(len(train))
lgb_test       = np.zeros(len(test))
lgb_local_test = np.zeros(len(test))
xgb_test       = np.zeros(len(test))
cat_test       = np.zeros(len(test))

lgb_scores = []; lgb_local_scores = []; xgb_scores = []; cat_scores = []

def make_X(df):
    for c in [col for col in ALL_FEATURES if col not in df.columns]:
        df[c] = np.nan
    return df[ALL_FEATURES].fillna(-999).astype(np.float32)

# Pre-compute test feature matrix (uses full-train aggregates)
X_te = make_X(test_f)

print(f"\n{'='*60}")
print("5-fold CV  —  fold-safe OOF | full-train features for test")
print(f"{'='*60}\n")

for fold, (tri, vai) in enumerate(kf.split(idx)):
    print(f"─── Fold {fold+1}/{N_FOLDS} ───")

    fold_tr = train.iloc[tri]
    fold_va = train.iloc[vai]

    # Fold-safe aggregates for OOF only
    fold_aggs = build_full_aggs(fold_tr)
    tr_f = apply_aggs(fold_tr, fold_aggs)
    va_f = apply_aggs(fold_va, fold_aggs)

    X_tr = make_X(tr_f); y_tr = y[tri]
    X_va = make_X(va_f); y_va = y[vai]

    # LightGBM main
    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(-1)])
    lgb_oof[vai]  = m_lgb.predict(X_va)
    lgb_test     += m_lgb.predict(X_te) / N_FOLDS
    s = r2_score(y_va, lgb_oof[vai]); lgb_scores.append(s)
    print(f"  LGB main   R²={s:.5f}  iter={m_lgb.best_iteration_}")

    # LightGBM local
    m_lgb_l = lgb.LGBMRegressor(**lgb_local_params)
    m_lgb_l.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(-1)])
    lgb_local_oof[vai]  = m_lgb_l.predict(X_va)
    lgb_local_test     += m_lgb_l.predict(X_te) / N_FOLDS
    s = r2_score(y_va, lgb_local_oof[vai]); lgb_local_scores.append(s)
    print(f"  LGB local  R²={s:.5f}  iter={m_lgb_l.best_iteration_}")

    # XGBoost
    m_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=250, eval_metric='rmse')
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    xgb_oof[vai]  = m_xgb.predict(X_va)
    xgb_test     += m_xgb.predict(X_te) / N_FOLDS
    s = r2_score(y_va, xgb_oof[vai]); xgb_scores.append(s)
    print(f"  XGB        R²={s:.5f}")

    # CatBoost
    m_cat = CatBoostRegressor(**cat_params, early_stopping_rounds=250)
    m_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True, verbose=0)
    cat_oof[vai]  = m_cat.predict(X_va)
    cat_test     += m_cat.predict(X_te) / N_FOLDS
    s = r2_score(y_va, cat_oof[vai]); cat_scores.append(s)
    print(f"  CAT        R²={s:.5f}\n")

# ── 9. OOF summary ───────────────────────────────────────────────────
lgb_s       = r2_score(y, lgb_oof)
lgb_local_s = r2_score(y, lgb_local_oof)
xgb_s       = r2_score(y, xgb_oof)
cat_s       = r2_score(y, cat_oof)

print(f"\n{'='*60}")
print("OOF Results (fold-safe — genuine estimate)")
print(f"{'='*60}")
print(f"  LGB main   {100*lgb_s:.3f}")
print(f"  LGB local  {100*lgb_local_s:.3f}")
print(f"  XGB        {100*xgb_s:.3f}")
print(f"  CAT        {100*cat_s:.3f}")

fold_arr = np.array(lgb_scores)
print(f"\n  LGB fold scores : {[f'{s:.4f}' for s in fold_arr]}")
print(f"  LGB fold std    : {fold_arr.std():.5f}")

# ── 10. Blending ─────────────────────────────────────────────────────
#  Weights: LGB main 40%, LGB local 18%, XGB 28%, CAT 14%
#  CAT gets a slight boost over the 90.77 code (12%→14%) because
#  with the exact gh_ts_mean feature it overfits less.
weights = np.array([0.40, 0.18, 0.28, 0.14])

oof_blend       = (weights[0]*lgb_oof + weights[1]*lgb_local_oof +
                   weights[2]*xgb_oof + weights[3]*cat_oof)
oof_blend_score = r2_score(y, oof_blend)

print(f"\n{'='*60}")
print("Ensemble blend")
print(f"{'='*60}")
print(f"  Weights: LGB_main={weights[0]}  LGB_local={weights[1]}  XGB={weights[2]}  CAT={weights[3]}")
print(f"  ► Ensemble OOF = {100*oof_blend_score:.3f}")

model_preds = (weights[0]*lgb_test + weights[1]*lgb_local_test +
               weights[2]*xgb_test + weights[3]*cat_test)

# ── 11. Final predictions: direct lookup blend ────────────────────────
#
#  Two-component final prediction:
#    A) gh_ts_mean from full train  = direct smoothed lookup (exact match)
#    B) model ensemble              = learned pattern interpolation
#
#  Where gh_ts_mean exists (has_exact=1): weight heavily toward lookup
#  Where it's missing (new timestamp):    rely on model
#
#  This gives near-perfect predictions for seen (geohash, timestamp)
#  combinations while gracefully handling unseen ones.
#
#  LOOKUP_WEIGHT controls the blend:
#    0.0 = pure model (90.77 baseline behaviour)
#    0.7 = 70% lookup + 30% model (recommended)
#    1.0 = pure lookup (like the 100-scorers, but only for exact matches)

LOOKUP_WEIGHT = 0.70

gh_ts_lookup   = test_f['gh_ts_mean'].values.astype(np.float64)
has_exact_mask = test_f['has_exact'].values.astype(bool)

final_preds = model_preds.copy()

# For rows with exact match: blend lookup + model
final_preds[has_exact_mask] = (
    LOOKUP_WEIGHT       * gh_ts_lookup[has_exact_mask] +
    (1 - LOOKUP_WEIGHT) * model_preds[has_exact_mask]
)
# For rows without exact match: use model only (already set above)

# Clip to observed demand range
demand_min = float(y.min())
demand_max = float(y.max())
final_preds = np.clip(final_preds, demand_min, demand_max)

print(f"\n{'='*60}")
print("Final prediction summary")
print(f"{'='*60}")
print(f"  Rows with exact gh×ts match : {has_exact_mask.sum()} / {len(test)}  ({has_exact_mask.mean():.1%})")
print(f"  Lookup weight (exact rows)  : {LOOKUP_WEIGHT}")
print(f"  Pred range : [{final_preds.min():.4f}, {final_preds.max():.4f}]")
print(f"  Pred mean  : {final_preds.mean():.4f}  (train mean: {y.mean():.4f})")

# ── 12. Submission ────────────────────────────────────────────────────
sub = pd.read_csv(sub_path)
assert sub.shape[0] == 41778, f"Expected 41778 rows, got {sub.shape[0]}"
assert len(final_preds) == len(sub)
assert 'demand' in sub.columns

sub['demand'] = final_preds
sub.to_csv('/content/submission.csv', index=False)

saved = pd.read_csv('/content/submission.csv')
assert saved.shape == (41778, 2)
assert saved['demand'].isna().sum() == 0, "Null predictions found!"

print(f"\n  Submission shape  : {saved.shape}  ✓")
print(f"  Null predictions  : {saved['demand'].isna().sum()}  ✓")
print("\nPreview:")
print(sub.head(10).to_string(index=False))
print("\nDownloading submission.csv ...")
files.download('/content/submission.csv')

# ═══════════════════════════════════════════════════════════════════
# APPENDIX: Tuning guide
# ═══════════════════════════════════════════════════════════════════
# If LB score < 93 after this:
#   → Raise LOOKUP_WEIGHT to 0.80 (trust direct lookup more)
#   → Lower ALPHA_GH_TS from 5 to 3 (trust raw means more)
#
# If LB score is 95-97 and you want to push to 98+:
#   → Raise LOOKUP_WEIGHT to 0.85-0.90
#   → Add gh × exact_day × ts_slot_15 aggregate (if train has enough obs)
#
# If OOF looks suspicious (> 99):
#   → Raise ALPHA_GH_TS to 10, lower LOOKUP_WEIGHT to 0.50
#
# LOOKUP_WEIGHT = 1.0 is equivalent to what the 100-scorers did
# (but only for rows where your train has the exact match).
# The model handles all other rows.

Upload your zip file...


Saving e88186124ec611f1.zip to e88186124ec611f1 (1).zip
Train: (77299, 11)  |  Test: (41778, 10)
Train columns: ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']
Static features done.
Global (non-target) aggregates merged.

5-fold CV  —  building fold-safe aggregates inside loop

─── Fold 1/5 ───
  LGB  R²=0.94120  (best iter=977)
  XGB  R²=0.93926
  CAT  R²=0.94445

─── Fold 2/5 ───
  LGB  R²=0.94069  (best iter=977)
  XGB  R²=0.93891
  CAT  R²=0.94475

─── Fold 3/5 ───
  LGB  R²=0.94240  (best iter=1002)
  XGB  R²=0.94094
  CAT  R²=0.94646

─── Fold 4/5 ───
  LGB  R²=0.93686  (best iter=1088)
  XGB  R²=0.93477
  CAT  R²=0.94171

─── Fold 5/5 ───
  LGB  R²=0.94003  (best iter=980)
  XGB  R²=0.93849
  CAT  R²=0.94565


OOF Results
  LGB  OOF R² = 0.94027  (score = 94.03)
  XGB  OOF R² = 0.93852  (score = 93.85)
  CAT  OOF R² = 0.94464  (score = 94.46)

Ensemble blend weights (fixed, LB-safe)
  LGB  we

ValueError: Length of values (41778) does not match length of index (5)

In [ ]:
import pandas as pd
from google.colab import files

final_preds=np.clip(final_preds,0,None)

sub=pd.DataFrame({
    'Index':range(len(final_preds)),
    'demand':final_preds
})

sub.to_csv('/content/submission.csv',index=False)

print(sub.head())
print(sub.shape)

files.download('/content/submission.csv')

   Index    demand
0      0  0.040712
1      1  0.035174
2      2  0.015617
3      3  0.043495
4      4  0.049858
(41778, 2)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>